Nhận diện vùng chứa biển số (LPD)

In [ ]:
# Khai báo thư viện
import os
import random
import shutil
import torch
import time
from google.colab import drive

# Khai báo phương thức
def count_png_files(directory):
  total_png = 0
  for root, dirs, files in os.walk(directory):
      png_in_current_folder = [f for f in files if f.lower().endswith('.png')]
      total_png += len(png_in_current_folder)
  return total_png

def safe_move_all(src, dst):
  if os.path.exists(src) and os.listdir(src):
      os.makedirs(dst, exist_ok=True)
      for f in os.listdir(src):
          shutil.move(os.path.join(src, f), os.path.join(dst, f))

# Khai báo đường dẫn folder
DRIVE_PATH   = "/content/drive/MyDrive/AI_Bien_So"
YOLO_PATH    = f"{DRIVE_PATH}/yolov5"
EXP_NAME     = "lpd_model"
DTS_NAME     = "lpd-dataset"
YAML_FILE    = "LP_detection.yaml"
CHECKPOINT   = f"{DRIVE_PATH}/runs/{EXP_NAME}/weights/last.pt"
ORIG_IMG_TRAIN = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}/images/train"
ORIG_LAB_TRAIN = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}/labels/train"
ORIG_IMG_VAL   = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}/images/val"
ORIG_LAB_VAL   = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}/labels/val"
MINI_IMG_TRAIN = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}_mini/images/train"
MINI_LAB_TRAIN = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}_mini/labels/train"
MINI_IMG_VAL   = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}_mini/images/val"
MINI_LAB_VAL   = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}_mini/labels/val"

# Test kết nối GPU của Server
!nvidia-smi

# Mount drive lên Server
if not os.path.exists("/content/drive"):
    drive.mount('/content/drive')

# Tạo folder dự án
if not os.path.exists(DRIVE_PATH):
  os.makedirs(DRIVE_PATH, exist_ok=True)
%cd $DRIVE_PATH

# Clone YOLOv5
if not os.path.exists('yolov5'):
  !git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -qr requirements.txt

# Tải yolov5m.pt
%cd /content/drive/MyDrive/AI_Bien_So/yolov5
if not os.path.exists('yolov5m.pt'):
  torch.hub.download_url_to_file('https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5m.pt', 'yolov5m.pt')

# Nhập số EPOCH muốn train
try:
    val = input("👉 Nhập TỔNG SỐ EPOCH mục tiêu (VD: 50, 100): ")
    TARGET_EPOCHS = int(val)
except:
    TARGET_EPOCHS = 50

# Sinh bộ dataset_mini ngẫu nhiên (hoàn thành đủ số epoch mới sinh bộ mới)
learned_epochs = 0
if os.path.exists(CHECKPOINT):
    ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
    learned_epochs = ckpt['epoch'] + 1

print(f"--- Hiện tại: {learned_epochs} Epochs | Mục tiêu: {TARGET_EPOCHS} ---")

if learned_epochs < TARGET_EPOCHS:
    root_dir = f"{DRIVE_PATH}/dataset_mini"
    mini_dataset_exists = os.path.exists(root_dir) and count_png_files(root_dir) > 0

    if not mini_dataset_exists:
        print("Đang chuẩn bị Mini Dataset để tối ưu tốc độ...")
        os.makedirs(MINI_IMG_TRAIN, exist_ok=True)
        os.makedirs(MINI_LAB_TRAIN, exist_ok=True)
        os.makedirs(MINI_IMG_VAL, exist_ok=True)
        os.makedirs(MINI_LAB_VAL, exist_ok=True)

        all_train_imgs = [f for f in os.listdir(ORIG_IMG_TRAIN) if f.lower().endswith('.jpg')]
        if all_train_imgs:
            selected = random.sample(all_train_imgs, min(1000, len(all_train_imgs)))
            for img in selected:
                lab = img.rsplit('.', 1)[0] + '.txt'
                if os.path.exists(os.path.join(ORIG_LAB_TRAIN, lab)):
                    shutil.move(os.path.join(ORIG_IMG_TRAIN, img), os.path.join(MINI_IMG_TRAIN, img))
                    shutil.move(os.path.join(ORIG_LAB_TRAIN, lab), os.path.join(MINI_LAB_TRAIN, lab))

        for f in os.listdir(ORIG_IMG_VAL):
            shutil.move(os.path.join(ORIG_IMG_VAL, f), os.path.join(MINI_IMG_VAL, f))
        for f in os.listdir(ORIG_LAB_VAL):
            shutil.move(os.path.join(ORIG_LAB_VAL, f), os.path.join(MINI_LAB_VAL, f))

        yaml_content = f"train: {MINI_IMG_TRAIN}\nval: {MINI_IMG_VAL}\nnc: 1\nnames: ['license_plate']"
        with open(f"{YOLO_PATH}/data/{YAML_FILE}", 'w') as f:
            f.write(yaml_content)

        print("Chờ Drive đồng bộ 45s trước khi Train...")
        time.sleep(45)

# Bắt đầu training
%cd {YOLO_PATH}
if learned_epochs < TARGET_EPOCHS:
    weights = CHECKPOINT if os.path.exists(CHECKPOINT) else "yolov5m.pt"
    !python train.py --img 320 --batch 16 --epochs {TARGET_EPOCHS} \
        --data data/{YAML_FILE} --weights {weights} \
        --project {DRIVE_PATH}/runs --name {EXP_NAME} --exist-ok --cache ram --patience 0
else:
    print(f"AI đã đạt đủ {learned_epochs} epochs.")

# Chuyển dữ liệu training về vị trí gốc
if mini_dataset_exists and learned_epochs >= TARGET_EPOCHS:
  if os.path.exists(MINI_IMG_TRAIN) and len(os.listdir(MINI_IMG_TRAIN)) > 0:
      print("\nĐang trả dữ liệu về vị trí gốc...")

      safe_move_all(MINI_IMG_TRAIN, ORIG_IMG_TRAIN)
      safe_move_all(MINI_LAB_TRAIN, ORIG_LAB_TRAIN)
      safe_move_all(MINI_IMG_VAL, ORIG_IMG_VAL)
      safe_move_all(MINI_LAB_VAL, ORIG_LAB_VAL)

      print("Đang đợi Google Drive đồng bộ (180 giây)...")
      time.sleep(180)

      if len(os.listdir(MINI_IMG_TRAIN)) == 0:
          shutil.rmtree(f"{DRIVE_PATH}/dataset_mini")
          yaml_content = f"train: {ORIG_IMG_TRAIN}\nval: {ORIG_IMG_VAL}\nnc: 1\nnames: ['license_plate']"
          with open(f"{YOLO_PATH}/data/{YAML_FILE}", 'w') as f:
              f.write(yaml_content)
          print("Đã dọn dẹp thư mục tạm.")

print("QUÁ TRÌNH TRAIN ĐÃ HOÀN THÀNH!")

# Chạy thử mô hình đã train lên tập val để xem kết quả
!python val.py --weights {DRIVE_PATH}/runs/{EXP_NAME}/weights/best.pt \
               --data data/{YAML_FILE} \
               --img 320 --half

Nhận diện ký tự trên biển số (LPR)

In [ ]:
# Khai báo thư viện
import os
import random
import shutil
import torch
import string
import time
from google.colab import drive

# Khai báo phương thức
def count_png_files(directory):
  total_png = 0
  for root, dirs, files in os.walk(directory):
      png_in_current_folder = [f for f in files if f.lower().endswith('.png')]
      total_png += len(png_in_current_folder)
  return total_png

def safe_move_all(src, dst):
  if os.path.exists(src) and os.listdir(src):
      os.makedirs(dst, exist_ok=True)
      for f in os.listdir(src):
          shutil.move(os.path.join(src, f), os.path.join(dst, f))

# Khai báo đường dẫn folder
DRIVE_PATH   = "/content/drive/MyDrive/AI_Bien_So"
YOLO_PATH    = f"{DRIVE_PATH}/yolov5"
EXP_NAME     = "lpr_model"
DTS_NAME     = "lpr-dataset"
YAML_FILE    = "LP_recognition.yaml"
CHECKPOINT   = f"{DRIVE_PATH}/runs/{EXP_NAME}/weights/last.pt"
ORIG_IMG_TRAIN = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}/images/train"
ORIG_LAB_TRAIN = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}/labels/train"
ORIG_IMG_VAL   = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}/images/val"
ORIG_LAB_VAL   = f"{DRIVE_PATH}/License-Plate/{DTS_NAME}/labels/val"

# Test kết nối GPU của Server
!nvidia-smi

# Mount drive lên Server
if not os.path.exists("/content/drive"):
    drive.mount('/content/drive')

# Tạo folder dự án
if not os.path.exists(DRIVE_PATH):
  os.makedirs(DRIVE_PATH, exist_ok=True)
%cd $DRIVE_PATH

# Clone YOLOv5
if not os.path.exists('yolov5'):
  !git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -qr requirements.txt

# Tải yolov5m.pt
%cd /content/drive/MyDrive/AI_Bien_So/yolov5
if not os.path.exists('yolov5m.pt'):
  torch.hub.download_url_to_file('https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5m.pt', 'yolov5m.pt')

# Tạo LP_recognition.yaml
names = list(string.digits + string.ascii_uppercase)
yaml_content = f"""
train: {ORIG_IMG_TRAIN}
val: {ORIG_IMG_VAL}
nc: {len(names)}
names: {names}
"""

with open(f"{YOLO_PATH}/data/{YAML_FILE}", 'w') as f:
    f.write(yaml_content.strip())

# Nhập số EPOCH muốn train
try:
    val = input("👉 Nhập TỔNG SỐ EPOCH mục tiêu (VD: 50, 100): ")
    TARGET_EPOCHS = int(val)
except:
    TARGET_EPOCHS = 50

# Lấy số epoch đã học (nếu có)
learned_epochs = 0
if os.path.exists(CHECKPOINT):
    ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
    learned_epochs = ckpt['epoch'] + 1

print(f"--- Hiện tại: {learned_epochs} Epochs | Mục tiêu: {TARGET_EPOCHS} ---")

# Bắt đầu training
%cd {YOLO_PATH}
if learned_epochs < TARGET_EPOCHS:
    weights = CHECKPOINT if os.path.exists(CHECKPOINT) else "yolov5m.pt"
    !python train.py --img 640 --batch 16 --epochs {TARGET_EPOCHS} \
        --data data/{YAML_FILE} --weights {weights} \
        --project {DRIVE_PATH}/runs --name {EXP_NAME} --exist-ok --cache ram --patience 0
else:
    print(f"AI đã đạt đủ {learned_epochs} epochs.")

print("QUÁ TRÌNH TRAIN ĐÃ HOÀN THÀNH!")

# Chạy thử mô hình đã train lên tập val để xem kết quả
!python val.py --weights {DRIVE_PATH}/runs/{EXP_NAME}/weights/best.pt \
               --data data/{YAML_FILE} \
               --img 640 --half

Sun Mar 22 06:17:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----